<a id="top"></a>
# House Sales: modellazione e MLflow (esercitazione – modulo 4)

Questa esercitazione riprende il dataset *House Sales* già usato nei moduli precedenti e introduce un flusso completo molto essenziale:

- caricamento del dataset nello stesso modo della scorsa esercitazione;
- addestramento di **due modelli di regressione** con relativa pipeline;
- **tracking con MLflow**, registrazione di un modello in produzione e caricamento dal *Model Registry*.

L’obiettivo non è complicare il modello, ma consolidare pochi passaggi chiave: pipeline semplici, metrica chiara, log su MLflow e recupero del modello registrato.


<a id="indice"></a>
## Indice

- [Obiettivi e prerequisiti](#obiettivi)
- [Caricamento del dataset House Sales](#caricamento-dati)
- [Preparazione dei dati per la regressione](#preparazione-dati)
- [Modello 1: Regressione lineare con pipeline](#modello1)
- [Modello 2: Random Forest con pipeline](#modello2)
- [Tracking dei modelli con MLflow](#mlflow-tracking)
- [Registrazione e caricamento dal Model Registry](#model-registry)
- [Riepilogo operativo](#riepilogo)


<a id="obiettivi"></a>
## Obiettivi e prerequisiti

L’esercitazione chiede esplicitamente tre azioni, tutte connesse alla parte di modellazione e MLOps del modulo 4:

1. **Leggere il dataset House Sales** esattamente come nella scorsa esercitazione.
2. **Preparare due modelli di regressione diversi**, ciascuno incapsulato in una pipeline scikit-learn semplice.
3. **Eseguire il log dei modelli su MLflow**, quindi registrare un modello come *Production* dalla UI e rileggerlo dal notebook.

Useremo:

- un **modello lineare** (`LinearRegression`);
- un **modello non lineare** a alberi (`RandomForestRegressor`).

<div style="background-color:#f4f4f4; padding:10px; border-left:4px solid #999;">

**Nota importante**  

Questa esercitazione dà già una versione completa del codice (“versione compilata”).  
Il compito dello studente è:

- leggere con calma i blocchi di istruzioni,
- collegarli ai singoli pezzi di codice,
- verificare che ogni azione (split, fit, log, registrazione) abbia uno scopo chiaro.

</div>


<a id="caricamento-dati"></a>
## Caricamento del dataset House Sales

Si utilizza lo stesso meccanismo del modulo precedente: il dataset *House Sales* viene scaricato da OpenML tramite `fetch_openml` e caricato in un `DataFrame` pandas.

<div style="background-color:#f4f4f4; padding:10px; border-left:4px solid #999;">

**Esercizio**  

1. Carica il dataset House Sales da OpenML usando `fetch_openml` con `data_id=42092` e `as_frame=True`.  
2. Estrai i dati in un `DataFrame` chiamato `house`.  
3. Verifica rapidamente quante righe e colonne ha il dataset e osserva le prime righe.

La cella di codice qui sotto contiene una possibile soluzione già compilata.

</div>

**Funzioni usate** (richiamo rapido):

- `fetch_openml` scarica un dataset da OpenML e restituisce dati e metadati.  
  Documentazione: <https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html>  
- `DataFrame.head` mostra le prime righe del `DataFrame`.  
  Documentazione: <https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html>


In [3]:
import pandas as pd
from sklearn.datasets import fetch_openml

# Caricamento del dataset House Sales da OpenML
raw = fetch_openml(data_id=42092, as_frame=True)
house = raw.frame.copy()

print("Shape del dataset (righe, colonne):", house.shape)
house.head()

Shape del dataset (righe, colonne): (21613, 20)


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


<a id="preparazione-dati"></a>
## Preparazione dei dati per la regressione

In questa esercitazione il target è il **prezzo di vendita** della casa.  
Il dataset su OpenML espone già un target, che possiamo leggere da `raw.target`. Useremo come feature solo le colonne numeriche, così la pipeline resta semplice.

Passaggi minimi:

1. Separare le feature `X` (solo numeriche) e il target `y`.  
2. Suddividere il dataset in train e test con `train_test_split`.  
3. Preparare l’elenco delle colonne numeriche su cui lavoreranno le pipeline.

<div style="background-color:#f4f4f4; padding:10px; border-left:4px solid #999;">

**Esercizio**  

- Crea `X` come sottoinsieme numerico dei dati.  
- Crea `y` a partire dal target del dataset e convertilo in `float`.  
- Esegui uno split train/test con test size al 20%, `random_state=42`.

La cella seguente mostra una soluzione completa.

</div>

**Funzioni chiave**:

- `train_test_split` suddivide i dati in training e test set.  
  Documentazione: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html>


In [4]:
from sklearn.model_selection import train_test_split

# X: solo feature numeriche
X = raw.data.select_dtypes(include=["number"]).copy()
# y: target del dataset (prezzo)
y = raw.target.astype(float)

print("Shape X:", X.shape)
print("Shape y:", y.shape)

# Suddivisione train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Shape X: (21613, 17)
Shape y: (21613,)
Train: (17290, 17) Test: (4323, 17)


<a id="modello1"></a>
## Modello 1: Regressione lineare con pipeline

Il primo modello è una **regressione lineare** (`LinearRegression`), incapsulata in una pipeline che:

1. **Imputa** eventuali valori mancanti con la mediana (`SimpleImputer`).  
2. **Scala** le feature (`StandardScaler`).  
3. Applica il modello lineare.

<div style="background-color:#f4f4f4; padding:10px; border-left:4px solid #999;">

**Esercizio**  

- Crea una pipeline scikit-learn con i passaggi: `imputer` → `scaler` → `model`.  
- Usa `SimpleImputer(strategy="median")` e `StandardScaler()`.  
- Come modello, usa `LinearRegression()`.  
- Fitta la pipeline sul train e valuta le prestazioni sul test con RMSE e R².

La cella seguente offre una possibile implementazione.

</div>

**Oggetti principali**:

- `Pipeline` collega in sequenza più step di preprocessing e modellazione.  
  Documentazione: <https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html>  
- `SimpleImputer` gestisce i valori mancanti con una strategia semplice (es. mediana).  
  Documentazione: <https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html>  
- `StandardScaler` standardizza le feature a media 0 e varianza 1.  
  Documentazione: <https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html>  
- `LinearRegression` implementa una regressione lineare ordinaria.  
  Documentazione: <https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html>


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error

# Pipeline per regressione lineare
lr_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

# Fit sul training set
lr_pipeline.fit(X_train, y_train)

# Valutazione sul test set
y_pred_lr = lr_pipeline.predict(X_test)
rmse_lr = root_mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Linear Regression - RMSE: {rmse_lr:.0f}")
print(f"Linear Regression - R²:   {r2_lr:.3f}")

Linear Regression - RMSE: 214473
Linear Regression - R²:   0.696


<a id="modello2"></a>
## Modello 2: Random Forest con pipeline

Il secondo modello è una **Random Forest di regressione** (`RandomForestRegressor`).  
Rispetto al modello lineare, permette di catturare relazioni non lineari senza modificare troppo la pipeline.

Per semplicità:

- riutilizziamo lo stesso schema di preprocessing (imputazione con mediana);
- non è necessario lo scaling per gli alberi, ma lasciarlo non crea problemi.

<div style="background-color:#f4f4f4; padding:10px; border-left:4px solid #999;">

**Esercizio**  

- Definisci una pipeline con `SimpleImputer` e `RandomForestRegressor`.  
- Imposta ad esempio `n_estimators=200` e `random_state=42`.  
- Allena il modello, valuta RMSE e R² e confrontali con la regressione lineare.

La soluzione qui sotto segue esattamente questi passi.

</div>

**Oggetti principali**:

- `RandomForestRegressor` costruisce un ensemble di alberi di regressione.  
  Documentazione: <https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html>


In [9]:
from sklearn.ensemble import RandomForestRegressor

# Pipeline per Random Forest
rf_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Fit sul training set
rf_pipeline.fit(X_train, y_train)

# Valutazione sul test set
y_pred_rf = rf_pipeline.predict(X_test)
rmse_rf = root_mean_squared_error(y_test, y_pred_rf)

r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest - RMSE: {rmse_rf:.0f}")
print(f"Random Forest - R²:   {r2_rf:.3f}")

Random Forest - RMSE: 147110
Random Forest - R²:   0.857


<a id="mlflow-tracking"></a>
## Tracking dei modelli con MLflow

Ora si passa al **tracking su MLflow**.  
L’obiettivo è:

- loggare entrambe le pipeline (lineare e Random Forest) come modelli MLflow;
- salvare metriche essenziali (RMSE, R²);
- preparare il terreno per la registrazione nel *Model Registry*.

Useremo un semplice store locale:

- `mlruns/` nella cartella corrente.

<div style="background-color:#f4f4f4; padding:10px; border-left:4px solid #999;">

**Esercizio**  

1. Imposta la tracking URI di MLflow su `file:./mlruns`.  
2. Definisci un esperimento chiamato `house_sales_regression`.  
3. Crea una funzione che, dato un nome modello e una pipeline, esegue:
   - `fit` della pipeline sui dati di training,
   - calcolo di RMSE e R² sul test,
   - log di parametri essenziali, metriche e modello.

La cella seguente implementa questa logica in modo compatto.

</div>

**API MLflow usate**:

- `mlflow.set_tracking_uri` imposta la posizione dove MLflow salva run e artefatti.  
  Documentazione: <https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.set_tracking_uri>  
- `mlflow.set_experiment` seleziona (o crea) un esperimento.  
  Documentazione: <https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.set_experiment>  
- `mlflow.start_run` apre un nuovo run di tracking.  
  Documentazione: <https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.start_run>  
- `mlflow.log_param`, `mlflow.log_metric` salvano parametri e metriche.  
  Documentazione: <https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_param>  
- `mlflow.sklearn.log_model` salva un modello scikit-learn come artefatto MLflow.  
  Documentazione: <https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html#mlflow.sklearn.log_model>


In [12]:
import mlflow
import mlflow.sklearn

# Imposta store locale e nome esperimento
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("house_sales_regression")


def train_and_log(model_name, pipeline, X_train, y_train, X_test, y_test):
    """Allena una pipeline, calcola metriche di base e logga tutto su MLflow."""
    with mlflow.start_run(run_name=model_name) as run:
        # Fit pipeline
        pipeline.fit(X_train, y_train)
        
        # Predizioni e metriche
        y_pred = pipeline.predict(X_test)
        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        # Log parametri minimi e metriche
        mlflow.log_param("model_type", model_name)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        
        # Log del modello
        mlflow.sklearn.log_model(pipeline, artifact_path="model")
        
        print(f"Run: {run.info.run_id}")
        print(f"  {model_name} - RMSE: {rmse:.0f}, R²: {r2:.3f}")
        return run.info.run_id


# Esegui il tracking per entrambi i modelli
run_id_lr = train_and_log("linear_regression", lr_pipeline, X_train, y_train, X_test, y_test)
run_id_rf = train_and_log("random_forest", rf_pipeline, X_train, y_train, X_test, y_test)

2025/11/21 17:26:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/21 17:26:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run: aa50e71969e24e128d67c52604503d84
  linear_regression - RMSE: 214473, R²: 0.696


2025/11/21 17:26:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/21 17:26:34 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run: 1c725270b9384bb3a0d4e6d1b6b9744a
  random_forest - RMSE: 147110, R²: 0.857


<a id="model-registry"></a>
## Registrazione e caricamento dal Model Registry

Una volta loggato il modello, si passa alla **registrazione** nel *Model Registry* tramite la UI di MLflow.  
L’obiettivo è avere un modello di riferimento marcato come **Production**, che poi verrà usato anche in una webapp Streamlit.

### Passaggi dalla UI di MLflow

<div style="background-color:#f4f4f4; padding:10px; border-left:4px solid #999;">

**Esercizio**  

1. Avvia la UI di MLflow (se non è già attiva):  
   ```bash
   mlflow ui --backend-store-uri file:./mlruns
   ```  
2. Apri il browser su `http://127.0.0.1:5000` (o l’indirizzo configurato).  
3. Vai all’esperimento **`house_sales_regression`**.  
4. Seleziona il run che corrisponde alla **Random Forest** (di solito con RMSE più basso).  
5. In alto a destra, usa **Register Model** e registra il modello con il nome **`house_sales_rf_prod`**.  
6. Nella sezione *Models*, apri `house_sales_rf_prod` e imposta la versione che ti interessa allo **stage `Production`**.

Questo nome (`house_sales_rf_prod`) verrà poi riutilizzato quando leggerai il modello da codice o da una webapp.

</div>

<div style="background-color:#fff8e1; padding:10px; border-left:4px solid #ffb300;">

**Attenzione**  

Assicurati di segnare:

- il nome del modello: `house_sales_rf_prod`;  
- lo *stage* impostato: `Production`.

Il caricamento da codice dipende esattamente da questi due elementi.

</div>

### Caricamento del modello registrato da notebook

Per leggere il modello dallo *Model Registry* useremo la API `mlflow.pyfunc.load_model`, indicando:

- il nome del modello (`house_sales_rf_prod`),
- lo stage (`Production`).

**Documentazione**: <https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html#mlflow.pyfunc.load_model>


In [ ]:
import mlflow.pyfunc

# Tracking URI coerente con quello usato per il training
mlflow.set_tracking_uri("file:./mlruns")

MODEL_NAME = "house_sales_rf_prod"
MODEL_STAGE = "Production"

model_uri = f"models:/{MODEL_NAME}/{MODEL_STAGE}"
loaded_model = mlflow.pyfunc.load_model(model_uri=model_uri)

print("Modello caricato dal registry:", MODEL_NAME, "- stage:", MODEL_STAGE)

# Esempio di predizione su un piccolo sottoinsieme del test set
sample = X_test.iloc[:5].copy()
predictions = loaded_model.predict(sample)

print("\nPredizioni (prime 5 righe):")
pd.DataFrame({
    "y_true": y_test.iloc[:5].values,
    "y_pred": predictions
})

<a id="riepilogo"></a>
## Riepilogo operativo

In questa esercitazione hai visto un flusso minimo ma completo per un problema di regressione su dati tabellari:

1. **Caricamento dati**: `fetch_openml` con `data_id=42092` per ottenere il dataset *House Sales* già noto dai moduli precedenti.  
2. **Preparazione dati**: selezione delle feature numeriche, split train/test e definizione di pipeline semplici.  
3. **Modellazione**: due pipeline di regressione distinte (modello lineare e Random Forest) con valutazione tramite RMSE e R².  
4. **MLOps di base**: tracking dei run con MLflow, log dei modelli come artefatti, registrazione del modello Random Forest come `house_sales_rf_prod` in stage `Production`.  
5. **Consumo del modello**: caricamento dal *Model Registry* via `mlflow.pyfunc.load_model` e predizione su nuovi dati.

Questa struttura è sufficiente per collegare, nel passo successivo, una webapp Streamlit che interroghi direttamente il modello registrato e lo usi per stimare il prezzo di vendita a partire dalle feature del dataset.
